# 2.0-feature-engineering

## Imports

In [65]:
import pandas as pd
from recruit_restaurant_visitor_forecasting.config.config import (
    AIR_AREA_COL,
    AIR_GENRE_COL,
    AIR_RESTAURANT_ID_COL,
    CALENDAR_DATE_COL,
    HPG_RESTAURANT_ID_COL,
    LATITUDE_COL,
    LONGITUDE_COL,
    VISIT_DATE_COL,
    VISITORS_COL,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
)
from recruit_restaurant_visitor_forecasting.config.features import (
    CITY_COL,
    DAY_OF_WEEK_COL,
    DAYS_OF_WEEK,
    OPEN_DATE_COL,
    RESERVE_AIR_COL,
    RESERVE_AIR_NBR_COL,
    RESERVE_HPG_COL,
    RESERVE_HPG_NBR_COL,
    TOTAL_RES_COL,
    TOTAL_RES_NBR_COL,
    VISITORS_NBR_COL,
    VISITORS_DOW,
    VISITORS_DOW_NBRS,
    RES_VISITORS_DIFF_COL,
    RES_VISITORS_DIFF_NBR_COL,
    GENRE_TE,
    AREA_TE,
    MEAN_PREF,
    MEDIAN_PREF,
    STD_PREF,
    DOW_WINDOW,
)
from recruit_restaurant_visitor_forecasting.dataset import (
    prepare_datetime_columns, standardize_date
)
from recruit_restaurant_visitor_forecasting.features import (
    add_basic_stats,
    add_golden_week_flg,
    add_dow_cum_agg,
    add_holiday_columns,
    add_lags,
    add_last_month_visitors,
    add_nbrs_reserves,
    add_neighbors_stats,
    add_opened_recently_flg,
    add_reserves_difference,
    add_sum_of_reserves,
    add_time_based_target_encoding,
    add_total_nbr_reservations,
    add_total_reservations,
    drop_first_month,
    add_open_usually_discr_rolling,
    add_reservation_impossibility,
    add_dow_rol_agg,
    remove_repetitions
)
from recruit_restaurant_visitor_forecasting.plots import plot_corr_matrix

In [66]:
air_visit_df = pd.read_csv(INTERIM_DATA_DIR / 'air_visit.csv')
air_reserve_df = pd.read_csv(INTERIM_DATA_DIR / 'air_reserve.csv')
hpg_reserve_df = pd.read_csv(INTERIM_DATA_DIR / 'hpg_reserve.csv', dtype={AIR_RESTAURANT_ID_COL: str})
future_df = pd.read_csv(INTERIM_DATA_DIR / 'sample_submission.csv')
air_store_df = pd.read_csv(INTERIM_DATA_DIR / 'air_store_info.csv')
hpg_store_df = pd.read_csv(INTERIM_DATA_DIR / 'hpg_store_info.csv')
date_info_df = pd.read_csv(RAW_DATA_DIR / 'date_info.csv')
store_rel_df = pd.read_csv(RAW_DATA_DIR / 'store_id_relation.csv')

In [67]:
prepare_datetime_columns(air_reserve_df)
prepare_datetime_columns(hpg_reserve_df)

standardize_date(date_info_df, CALENDAR_DATE_COL)
standardize_date(air_visit_df, VISIT_DATE_COL)
standardize_date(future_df, VISIT_DATE_COL)

## Features

### Air & hpg stores

In [68]:
air_store_df = air_store_df.drop([AIR_AREA_COL, LATITUDE_COL, LONGITUDE_COL], axis=1)

### Reserve dataframes

For further work, it is necessary to know the total number of reservations in the restaurant per day.

In [69]:
hpg_reserve_df = remove_repetitions(air_reserve_df, hpg_reserve_df)

In [70]:
air_res_sum = add_sum_of_reserves(air_reserve_df, RESERVE_AIR_COL, max_res_diff=39)
air_res_sum = air_res_sum.merge(
    air_store_df[[AIR_RESTAURANT_ID_COL, CITY_COL]], on=AIR_RESTAURANT_ID_COL
)
air_res_sum = add_nbrs_reserves(air_res_sum, RESERVE_AIR_COL, CITY_COL, RESERVE_AIR_NBR_COL)
air_res_sum.head()

,air_store_id,visit_date,air_reserves,air_reserves_0,air_reserves_1,air_reserves_2,air_reserves_3,air_reserves_4,air_reserves_5,air_reserves_6,...,air_reserves_31,air_reserves_32,air_reserves_33,air_reserves_34,air_reserves_35,air_reserves_36,air_reserves_37,air_reserves_38,city,air_reserves_nbrs
0,air_00a91d42b08b08d9,2016-10-31,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Tōkyō-to,7.829268
1,air_00a91d42b08b08d9,2016-12-05,9,9.0,9.0,9.0,9.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Tōkyō-to,9.754717
2,air_00a91d42b08b08d9,2016-12-14,18,18.0,18.0,18.0,18.0,18.0,18.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Tōkyō-to,13.726027
3,air_00a91d42b08b08d9,2016-12-17,2,2.0,2.0,2.0,2.0,2.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Tōkyō-to,20.884211
4,air_00a91d42b08b08d9,2016-12-20,4,4.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Tōkyō-to,14.944444


In [71]:
air_res_sum[RESERVE_AIR_NBR_COL].isna().sum()

np.int64(0)

However, air_reserve dataframe still has a large number of gaps.

In [72]:
hpg_res_sum = add_sum_of_reserves(hpg_reserve_df, RESERVE_HPG_COL, HPG_RESTAURANT_ID_COL, max_res_diff=39)
hpg_res_sum = hpg_res_sum.merge(
    hpg_store_df[[HPG_RESTAURANT_ID_COL, CITY_COL]], on=HPG_RESTAURANT_ID_COL
)
hpg_res_sum = add_nbrs_reserves(hpg_res_sum, RESERVE_HPG_COL, CITY_COL, RESERVE_HPG_NBR_COL)
hpg_res_sum.head()

,hpg_store_id,visit_date,hpg_reserves,hpg_reserves_0,hpg_reserves_1,hpg_reserves_2,hpg_reserves_3,hpg_reserves_4,hpg_reserves_5,hpg_reserves_6,...,hpg_reserves_31,hpg_reserves_32,hpg_reserves_33,hpg_reserves_34,hpg_reserves_35,hpg_reserves_36,hpg_reserves_37,hpg_reserves_38,city,hpg_reserves_nbrs
0,hpg_001ce40a1f873e4f,2016-01-13,4,4.0,4.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ōsaka-fu,5.513043
1,hpg_001ce40a1f873e4f,2016-01-27,7,7.0,7.0,7.0,7.0,7.0,7.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ōsaka-fu,6.202614
2,hpg_001ce40a1f873e4f,2016-02-13,2,2.0,2.0,2.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ōsaka-fu,6.532915
3,hpg_001ce40a1f873e4f,2016-02-27,8,8.0,8.0,8.0,8.0,8.0,8.0,8.0,...,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,Ōsaka-fu,7.452012
4,hpg_001ce40a1f873e4f,2016-03-16,2,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ōsaka-fu,7.291892


In [73]:
hpg_res_sum[RESERVE_HPG_NBR_COL].isna().sum()

np.int64(0)

In [74]:
hpg_res_sum_mapped = hpg_res_sum.merge(store_rel_df, on=HPG_RESTAURANT_ID_COL)

### Date info

It is necessary to add a feature for the distance to the nearest holiday.

In [75]:
date_info_df = add_holiday_columns(date_info_df, CALENDAR_DATE_COL)

It is also necessary to designate Golden Week, since not all days of this week are holidays.

In [76]:
date_info_df = add_golden_week_flg(date_info_df, [2016, 2017], CALENDAR_DATE_COL)

Later we will need to match the weekday of a selected date with the restaurant’s open/closed flag for that weekday, so it makes sense to rename the days.

In [77]:
FULL_WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
mapping = dict(zip(FULL_WEEKDAYS, DAYS_OF_WEEK))
date_info_df[DAY_OF_WEEK_COL] = date_info_df[DAY_OF_WEEK_COL].replace(mapping)

### Air visit

#### Start dates

We can consider the restaurant's opening date. Since simply counting the number of days since opening may not be sufficient due to the increase in days over time, it's best to flag the restaurant's opening as occurring within the last six months. A **potential issue**: some restaurants either planned to open earlier than their minimum opening date but didn't, or didn't report visitors for earlier dates. This is indicated by the fact that air_reserve dataframe has reservations for earlier dates.

In [78]:
air_open_dates = (
    air_visit_df.groupby(AIR_RESTAURANT_ID_COL)[VISIT_DATE_COL]
    .min()
    .rename(OPEN_DATE_COL)
)

In [79]:
air_visit_df = add_opened_recently_flg(
    air_visit_df, air_open_dates, VISIT_DATE_COL, AIR_RESTAURANT_ID_COL
)
future_df = add_opened_recently_flg(
    future_df, air_open_dates, VISIT_DATE_COL, AIR_RESTAURANT_ID_COL
)

In [80]:
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1


In [81]:
future_df.head()

,id,visitors,air_store_id,visit_date,open_date,opened_recently
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0


#### Open dates

In [82]:
air_visit_df = (
    air_visit_df.merge(date_info_df, left_on=VISIT_DATE_COL, right_on=CALENDAR_DATE_COL)
    .merge(air_store_df, on=AIR_RESTAURANT_ID_COL)
    .drop(CALENDAR_DATE_COL, axis=1)
)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,city
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,Tōkyō-to
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,Tōkyō-to
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,Tōkyō-to
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,Tōkyō-to
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,Tōkyō-to


In [83]:
air_visit_df = add_open_usually_discr_rolling(air_visit_df, 0.5)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,city,open_usually
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,Tōkyō-to,NaN
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,Tōkyō-to,NaN
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,Tōkyō-to,NaN
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,Tōkyō-to,NaN
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,Tōkyō-to,NaN


Open_usually feature contains the probability that a restaurant will be open on a given day. It depends on the holiday feature, the percentage of non-zeros on holidays, and on individual days of the week.

In [84]:
future_df = (
    future_df.merge(date_info_df, left_on=VISIT_DATE_COL, right_on=CALENDAR_DATE_COL)
    .merge(air_store_df, on=AIR_RESTAURANT_ID_COL)
    .drop(CALENDAR_DATE_COL, axis=1)
)

#### Mean visitors by air city.

It is necessary to make smoothed target encoding, while the average value will be used for new areas. In this case, it is worth considering only days when the number of visitors is not zero, so that only open restaurants are taken into account.

In [85]:
air_visit_df, future_df = add_time_based_target_encoding(
    air_visit_df, future_df, CITY_COL, VISITORS_COL, AREA_TE
)

In [86]:
air_visit_df[
    [AIR_RESTAURANT_ID_COL, VISITORS_COL, AREA_TE, CITY_COL, VISIT_DATE_COL]
].head()

,air_store_id,visitors,air_city_te,city,visit_date
0,air_00a91d42b08b08d9,35,20.497009,Tōkyō-to,2016-07-01
1,air_00a91d42b08b08d9,9,20.568186,Tōkyō-to,2016-07-02
2,air_00a91d42b08b08d9,0,20.634902,Tōkyō-to,2016-07-03
3,air_00a91d42b08b08d9,20,20.643692,Tōkyō-to,2016-07-04
4,air_00a91d42b08b08d9,25,20.590954,Tōkyō-to,2016-07-05


In [87]:
future_df[[AIR_RESTAURANT_ID_COL, AREA_TE, CITY_COL, VISIT_DATE_COL]].head()

,air_store_id,air_city_te,city,visit_date
0,air_00a91d42b08b08d9,19.279498,Tōkyō-to,2017-04-23
1,air_00a91d42b08b08d9,19.279498,Tōkyō-to,2017-04-24
2,air_00a91d42b08b08d9,19.279498,Tōkyō-to,2017-04-25
3,air_00a91d42b08b08d9,19.279498,Tōkyō-to,2017-04-26
4,air_00a91d42b08b08d9,19.279498,Tōkyō-to,2017-04-27


#### Mean visitors by air genre

It is necessary to make smoothed target encoding, while the average value will be used for new genres.

In [88]:
air_visit_df, future_df = add_time_based_target_encoding(
    air_visit_df, future_df, AIR_GENRE_COL, VISITORS_COL, GENRE_TE
)

In [89]:
air_visit_df[
    [AIR_RESTAURANT_ID_COL, VISITORS_COL, GENRE_TE, AIR_GENRE_COL, VISIT_DATE_COL]
].head()

,air_store_id,visitors,air_genre_te,air_genre_name,visit_date
0,air_00a91d42b08b08d9,35,21.216062,Italian/French,2016-07-01
1,air_00a91d42b08b08d9,9,21.279619,Italian/French,2016-07-02
2,air_00a91d42b08b08d9,0,21.352360,Italian/French,2016-07-03
3,air_00a91d42b08b08d9,20,21.376200,Italian/French,2016-07-04
4,air_00a91d42b08b08d9,25,21.334206,Italian/French,2016-07-05


In [90]:
future_df[[AIR_RESTAURANT_ID_COL, GENRE_TE, AIR_GENRE_COL, VISIT_DATE_COL]].head()

,air_store_id,air_genre_te,air_genre_name,visit_date
0,air_00a91d42b08b08d9,20.969311,Italian/French,2017-04-23
1,air_00a91d42b08b08d9,20.969311,Italian/French,2017-04-24
2,air_00a91d42b08b08d9,20.969311,Italian/French,2017-04-25
3,air_00a91d42b08b08d9,20.969311,Italian/French,2017-04-26
4,air_00a91d42b08b08d9,20.969311,Italian/French,2017-04-27


#### Total reserved visitors

Total reserved visitors (from air_reserve and hpg_reserve) on this day - for this restaurant/for neighbors.

By neighbors, we designate restaurants that are located in the same city.

In [92]:
air_visit_df = add_reservation_impossibility(air_visit_df, air_res_sum, hpg_res_sum_mapped)
air_visit_df = add_total_reservations(air_visit_df, air_res_sum, hpg_res_sum_mapped, CITY_COL, max_res_diff=39)
air_visit_df = add_total_nbr_reservations(air_visit_df, hpg_res_sum, CITY_COL).fillna(0)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,total_reservations_31,total_reservations_32,total_reservations_33,total_reservations_34,total_reservations_35,total_reservations_36,total_reservations_37,total_reservations_38,total_reservations_39,total_reservations_nbrs
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.305774
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.821683
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.256637
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.464730
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.799401


In [93]:
future_df = add_reservation_impossibility(future_df, air_res_sum, hpg_res_sum_mapped)
future_df = add_total_reservations(future_df, air_res_sum, hpg_res_sum_mapped, CITY_COL, max_res_diff=39)
future_df = add_total_nbr_reservations(future_df, hpg_res_sum, CITY_COL)
future_df.head()

,id,visitors,air_store_id,visit_date,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,...,total_reservations_31,total_reservations_32,total_reservations_33,total_reservations_34,total_reservations_35,total_reservations_36,total_reservations_37,total_reservations_38,total_reservations_39,total_reservations_nbrs
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0,Sun,0,-6,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.325949
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0,Mon,0,-5,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.690802
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0,Tue,0,-4,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.652330
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0,Wed,0,-3,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.196581
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0,Th,0,-2,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.380583


#### Rolling mean/median/std of visitors

If open_usually == 0 and visitors == 0, the value is replaced with NaN. Otherwise, zero is included in the aggregation. Later, we need to take care of the initial rows of the window, because right now they contain non-NaN values, even though conceptually they should be NaN.

In [94]:
aggs = [
    ("mean", {}),
    ("median", {}),
    ("std", {"ddof": 0}),
    ("max", {}),
    ("min", {}),
]
air_visit_df = add_basic_stats(air_visit_df, VISITORS_COL, AIR_RESTAURANT_ID_COL, aggs=aggs)
air_visit_df = add_neighbors_stats(air_visit_df, VISITORS_COL, CITY_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_nbrs,visitors_nbrs_mean_7,visitors_nbrs_median_7,visitors_nbrs_std_7,visitors_nbrs_mean_14,visitors_nbrs_median_14,visitors_nbrs_std_14,visitors_nbrs_mean_28,visitors_nbrs_median_28,visitors_nbrs_std_28
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,...,25.165939,21.177588,20.50838,3.113258,20.661869,20.498634,3.330588,20.155665,19.507014,3.427392
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,...,25.169683,21.318805,20.50838,3.264856,20.676396,20.498634,3.349703,20.247613,19.507014,3.524657
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,...,21.453172,21.340729,20.50838,3.290029,20.687358,20.498634,3.364107,20.297345,19.507014,3.583441
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,...,16.325459,20.976897,20.50838,3.111829,20.670038,20.498634,3.359493,20.290875,19.507014,3.581184
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,...,17.797753,20.964567,20.50838,3.130010,20.700804,20.498634,3.317328,20.338693,19.507014,3.518429


#### Visitors lag features

In [95]:
air_visit_df = add_lags(air_visit_df, AIR_RESTAURANT_ID_COL, VISITORS_COL)
air_visit_df = add_lags(air_visit_df, CITY_COL, VISITORS_NBR_COL, True)
air_visit_df.tail()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_nbrs_std_14,visitors_nbrs_mean_28,visitors_nbrs_median_28,visitors_nbrs_std_28,visitors_lag_1,visitors_lag_7,visitors_lag_28,visitors_nbrs_lag_1,visitors_nbrs_lag_7,visitors_nbrs_lag_28
283173,air_f267dd70a6a6b5d3,2017-04-22,44,2016-07-01,0,Sat,0,-7,0,Creative cuisine,...,5.184765,21.450524,20.700382,5.070385,9.0,59.0,60.0,24.641221,28.853846,32.387597
289253,air_f927b2da69a82341,2017-04-22,7,2016-01-04,0,Sat,0,-7,0,Japanese food,...,5.184765,21.450524,20.700382,5.070385,13.0,9.0,14.0,24.641221,28.853846,32.387597
294159,air_fdc02ec4a3d21ea4,2017-04-22,9,2016-07-01,0,Sat,0,-7,0,Dining bar,...,5.184765,21.450524,20.700382,5.070385,4.0,26.0,32.0,24.641221,28.853846,32.387597
295999,air_fee8dcf4d619598e,2017-04-22,53,2016-07-01,0,Sat,0,-7,0,Italian/French,...,5.184765,21.450524,20.700382,5.070385,27.0,47.0,45.0,24.641221,28.853846,32.387597
296295,air_fef9ccb3ba0da2f7,2017-04-22,5,2016-07-01,0,Sat,0,-7,0,Japanese food,...,5.184765,21.450524,20.700382,5.070385,3.0,23.0,12.0,24.641221,28.853846,32.387597


#### Visitors on the same day last month


The feature has a drawback: if the previous month had 30 days and the current month has 31, then the feature for the 31st day will correspond to the 30th day. If there were 0 visitors, and the open_usually flag is 0, then 0 visitors are indicated.

In [96]:
air_visit_df = add_last_month_visitors(air_visit_df)
air_visit_df.tail()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_nbrs_mean_28,visitors_nbrs_median_28,visitors_nbrs_std_28,visitors_lag_1,visitors_lag_7,visitors_lag_28,visitors_nbrs_lag_1,visitors_nbrs_lag_7,visitors_nbrs_lag_28,visitors_last_month
296883,air_f267dd70a6a6b5d3,2017-04-22,44,2016-07-01,0,Sat,0,-7,0,Creative cuisine,...,21.450524,20.700382,5.070385,9.0,59.0,60.0,24.641221,28.853846,32.387597,5.0
296884,air_f927b2da69a82341,2017-04-22,7,2016-01-04,0,Sat,0,-7,0,Japanese food,...,21.450524,20.700382,5.070385,13.0,9.0,14.0,24.641221,28.853846,32.387597,0.0
296885,air_fdc02ec4a3d21ea4,2017-04-22,9,2016-07-01,0,Sat,0,-7,0,Dining bar,...,21.450524,20.700382,5.070385,4.0,26.0,32.0,24.641221,28.853846,32.387597,1.0
296886,air_fee8dcf4d619598e,2017-04-22,53,2016-07-01,0,Sat,0,-7,0,Italian/French,...,21.450524,20.700382,5.070385,27.0,47.0,45.0,24.641221,28.853846,32.387597,27.0
296887,air_fef9ccb3ba0da2f7,2017-04-22,5,2016-07-01,0,Sat,0,-7,0,Japanese food,...,21.450524,20.700382,5.070385,3.0,23.0,12.0,24.641221,28.853846,32.387597,3.0


#### Historical day-of-week mean of visitors


In [97]:
aggs = [MEAN_PREF, MEDIAN_PREF, STD_PREF]
air_visit_df, future_df = add_dow_cum_agg(
    air_visit_df, VISITORS_COL, VISITORS_DOW, aggs, future_df
)
air_visit_df, future_df = add_dow_cum_agg(
    air_visit_df, VISITORS_NBR_COL, VISITORS_DOW_NBRS, aggs, future_df
)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_nbrs_lag_1,visitors_nbrs_lag_7,visitors_nbrs_lag_28,visitors_last_month,visitors_dow_mean,visitors_dow_median,visitors_dow_std,visitors_nbrs_dow_mean,visitors_nbrs_dow_median,visitors_nbrs_dow_std
0,air_05c325d315cc17f5,2016-01-01,29,2016-01-01,1,Fri,1,0,0,Izakaya,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,air_09a845d5b5944b01,2016-01-01,56,2016-01-01,1,Fri,1,0,0,Izakaya,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,air_298513175efdf261,2016-01-01,12,2016-01-01,1,Fri,1,0,0,Cafe/Sweets,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,air_3b6438b125086430,2016-01-01,10,2016-01-01,1,Fri,1,0,0,Bar/Cocktail,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,air_60a7057184ec7ec7,2016-01-01,64,2016-01-01,1,Fri,1,0,0,Izakaya,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [98]:
future_df.head()

,id,visitors,air_store_id,visit_date,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,...,total_reservations_37,total_reservations_38,total_reservations_39,total_reservations_nbrs,visitors_dow_mean,visitors_dow_median,visitors_dow_std,visitors_nbrs_dow_mean,visitors_nbrs_dow_median,visitors_nbrs_dow_std
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0,Sun,0,-6,0,...,0.0,0.0,0.0,7.325949,0.048780,0.0,0.312348,20.563693,20.940828,3.180149
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0,Mon,0,-5,0,...,0.0,0.0,0.0,9.690802,18.707317,18.0,12.209103,14.778063,14.923645,2.252889
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0,Tue,0,-4,0,...,0.0,0.0,0.0,8.652330,22.902439,24.0,10.261103,16.148191,16.326039,2.561193
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0,Wed,0,-3,0,...,0.0,0.0,0.0,11.196581,27.024390,28.0,10.588881,18.153140,18.086093,2.468991
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0,Th,0,-2,0,...,0.0,0.0,0.0,9.380583,26.756098,29.0,11.173139,17.857655,17.967320,2.280091


In [99]:
air_visit_df = add_dow_rol_agg(air_visit_df, VISITORS_COL, VISITORS_DOW, aggs, DOW_WINDOW)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_last_month,visitors_dow_mean,visitors_dow_median,visitors_dow_std,visitors_nbrs_dow_mean,visitors_nbrs_dow_median,visitors_nbrs_dow_std,visitors_dow_mean_4,visitors_dow_median_4,visitors_dow_std_4
0,air_05c325d315cc17f5,2016-01-01,29,2016-01-01,1,Fri,1,0,0,Izakaya,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,air_09a845d5b5944b01,2016-01-01,56,2016-01-01,1,Fri,1,0,0,Izakaya,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,air_298513175efdf261,2016-01-01,12,2016-01-01,1,Fri,1,0,0,Cafe/Sweets,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,air_3b6438b125086430,2016-01-01,10,2016-01-01,1,Fri,1,0,0,Bar/Cocktail,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,air_60a7057184ec7ec7,2016-01-01,64,2016-01-01,1,Fri,1,0,0,Izakaya,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Rolling reserve/visitors difference


In [100]:
air_visit_df = add_reserves_difference(
    air_visit_df, VISITORS_COL, TOTAL_RES_COL, RES_VISITORS_DIFF_COL
)
air_visit_df = add_reserves_difference(
    air_visit_df, VISITORS_NBR_COL, TOTAL_RES_NBR_COL, RES_VISITORS_DIFF_NBR_COL
)

In [101]:
aggs = [("mean", {})]
air_visit_df = add_basic_stats(
    air_visit_df, RES_VISITORS_DIFF_COL, AIR_RESTAURANT_ID_COL, aggs
)
air_visit_df = add_neighbors_stats(
    air_visit_df, RES_VISITORS_DIFF_NBR_COL, CITY_COL, aggs, False
)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_dow_median_4,visitors_dow_std_4,res_visitors_diff,res_visitors_diff_mean_7,res_visitors_diff_mean_14,res_visitors_diff_mean_28,nbr_res_visitors_diff,nbr_res_visitors_diff_mean_7,nbr_res_visitors_diff_mean_14,nbr_res_visitors_diff_mean_28
106737,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,...,NaN,NaN,NaN,0.0,0.0,0.0,-12.298967,-12.416233,-12.545709,-12.612287
107198,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,...,NaN,NaN,-35.0,0.0,0.0,0.0,-16.571661,-12.839972,-12.642776,-12.619208
107664,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,...,NaN,NaN,-9.0,-35.0,-35.0,-35.0,-17.056154,-13.289772,-12.696017,-12.709705
108131,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,...,NaN,NaN,0.0,-22.0,-22.0,-22.0,-15.042359,-13.398031,-12.715224,-12.735764
108600,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,...,NaN,NaN,-20.0,-22.0,-22.0,-22.0,-9.841539,-13.131959,-12.717024,-12.751523


## Feature correlation

In [ ]:
corr = air_visit_df.corr(numeric_only=True)
plot_corr_matrix(corr)

## Saving

#### NaN dropping

In [103]:
air_visit_df.to_csv(PROCESSED_DATA_DIR / "air_visit.csv", index=False)
air_visit_df = drop_first_month(air_visit_df)
air_visit_df = air_visit_df.sort_values(VISIT_DATE_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_dow_median_4,visitors_dow_std_4,res_visitors_diff,res_visitors_diff_mean_7,res_visitors_diff_mean_14,res_visitors_diff_mean_28,nbr_res_visitors_diff,nbr_res_visitors_diff_mean_7,nbr_res_visitors_diff_mean_14,nbr_res_visitors_diff_mean_28
78862,air_8d50c64692322dff,2016-02-01,0,2016-01-01,1,Mon,0,-10,0,Bar/Cocktail,...,4.5,10.862780,-5.0,-3.333333,-4.750000,-5.153846,-15.186270,-12.23017,-12.047024,-13.255695
251509,air_35c4732dcbfe31be,2016-02-01,0,2016-01-01,1,Mon,0,-10,0,Dining bar,...,8.5,2.160247,-18.0,-7.333333,-8.750000,-8.640000,-21.563378,-13.82166,-13.427962,-14.050825
251520,air_536043fcf1a4f8a4,2016-02-01,33,2016-01-01,1,Mon,0,-10,0,Bar/Cocktail,...,28.0,18.025445,-32.0,-25.714286,-26.928571,-28.384615,-21.563378,-13.82166,-13.427962,-14.050825
251511,air_39dccf7df20b1c6a,2016-02-01,26,2016-01-01,1,Mon,0,-10,0,Izakaya,...,26.0,6.500000,-38.0,-18.714286,-21.928571,-23.571429,-21.563378,-13.82166,-13.427962,-14.050825
78792,air_36bcf77d3382d36e,2016-02-01,17,2016-01-01,1,Mon,0,-10,0,Bar/Cocktail,...,20.5,19.200694,-56.0,-29.857143,-28.000000,-28.814815,-15.186270,-12.23017,-12.047024,-13.255695


In [104]:
air_visit_df = air_visit_df.drop(columns=[OPEN_DATE_COL])
labels = air_visit_df[VISITORS_COL].copy()
features = air_visit_df.drop(columns=[VISITORS_COL])
future_df = future_df.drop(columns=[OPEN_DATE_COL, "id", VISITORS_COL])
future_df[TOTAL_RES_NBR_COL] = future_df[TOTAL_RES_NBR_COL].fillna(0)
features.to_csv(PROCESSED_DATA_DIR / "features.csv", index=False)
labels.to_csv(PROCESSED_DATA_DIR / "labels.csv", index=False)
future_df.to_csv(PROCESSED_DATA_DIR / "test_features.csv", index=False)

## Conclusion

### Added features.

- Operating schedule - regular working days, extracted from the regular gaps in the data frame.
- Days to/from the nearest holiday (negative is days until, positive is days after).
- Holiday indicator (currently included in date_info).
- Separate indicators for Golden Week dates.

- Number of visitors on this day last month for this restaurant.

- Indicator of whether the restaurant has been open within the last 6 months.
- Days since last recorded visit for air_visit dataframe.

- Rolling mean/median/std of visitors over the past week, month - for this restaurant/for neighbors.
- Historical day-of-week mean up to (but not including) the current day - for this restaurant/for neighbors.
- Rolling reserve/visitors difference over the past 7 / 28 days - for this restaurant/for neighbors.
- Lag 1, 7, 28 of the number of visitors - for this restaurant/for neighbors.

- Smoothed target encoding of visitors by air_area_name.
- Smoothed target encoding of visitors by air_genre_name.

- Total reserved visitors (from air_reserve and hpg_reserve) on this day - for this restaurant/for neighbors.

### Features planned for addition.

- Days since last recorded visit for sample_submission.

### Possible features.

- Daily temperature - try to get the weather forecast.
- Precipitation probability.
- Hpg genre.

Decomposition features:
- Trend.
- Trend difference for last month.
- Seasonal.
- Residual mean for last month.